# Phase 8 — INT8 post-training quantization (TensorFlow Lite)
* Models: for each configuration, the teacher (final Model E, Phase 5) and the student with knowledge
  distillation (Phase 7). The merged configuration is the main model of the thesis; the other two are for comparison.
* Method: post-training full-integer quantization with TensorFlow Lite — weights and activations converted from
  32-bit floating point to 8-bit integers, without retraining (Umar et al., 2025; Jacob et al., 2018).
  Activation ranges are calibrated on a representative dataset of 500 records drawn from the TRAINING split,
  stratified by class; the guide recommends a small subset of about 100-500 training or validation records
  (Google AI Edge, n.d., post-training quantization guide).
* Each model is evaluated three ways on the complete test split: Keras float32, TFLite float32, TFLite INT8, so the
  effect of the conversion and of quantization can be separated. Reported: Macro F1, accuracy, weighted F1,
  per-class F1, model file size, and CPU latency per record measured as in the single-stream scenario of MLPerf
  Inference (Reddi et al., 2020): one record per query, 1,024 queries, 90th-percentile latency (mean also reported);
  untimed warm-up runs precede the measurement.
* Runs on a CPU runtime. Outputs: outputs_v3/int8_v1/ (models/, results/, summary.json)

In [ ]:
# %% Cell 0 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %% Cell 1 — settings and helpers
import os, json, time, gc, platform
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

FEAT  = '/content/drive/MyDrive/Thesis/outputs_v3/features_v3/'
FINAL = '/content/drive/MyDrive/Thesis/outputs_v3/final_v1/models/'
KD    = '/content/drive/MyDrive/Thesis/outputs_v3/kd_v1/models/'
OUT   = '/content/drive/MyDrive/Thesis/outputs_v3/int8_v1/'
for sub in ('models/', 'results/'):
    os.makedirs(OUT + sub, exist_ok=True)
SEED, N_REP, N_LAT, N_WARM, BATCH = 42, 500, 1024, 100, 8192
CLASS_NAMES = ['BENIGN', 'Botnet', 'BruteForce', 'DDoS', 'DoS', 'PortScan', 'Spoofing', 'WebAttack']
CONFIGS = ['config1_merged', 'config2_cicids2017', 'config3_ciciot2023']
MODELS = [(c, 'teacher', FINAL + f'final_{c}.keras') for c in CONFIGS] + [(c, 'student_kd', KD + f'kd_{c}.keras') for c in CONFIGS]
MODELS = [m for m in MODELS if m[0] == 'config1_merged'] + [m for m in MODELS if m[0] != 'config1_merged']   # merged first

try:
    Interpreter = tf.lite.Interpreter
except AttributeError:
    os.system('pip -q install ai-edge-litert'); from ai_edge_litert.interpreter import Interpreter

def log(msg):
    print(time.strftime('%Y-%m-%d %H:%M:%S'), msg, flush=True)

def metrics(y, pred):
    labels = np.unique(y)
    f1c = f1_score(y, pred, average=None, labels=labels, zero_division=0)
    return dict(macro_f1=float(f1c.mean()), accuracy=float(accuracy_score(y, pred)),
                weighted_f1=float(f1_score(y, pred, average='weighted', labels=labels, zero_division=0)),
                per_class_f1={CLASS_NAMES[k]: round(float(v), 4) for k, v in zip(labels, f1c)})

def tflite_predict(path, X):
    it = Interpreter(model_path=path); inp, out = it.get_input_details()[0], it.get_output_details()[0]
    scale, zp = inp['quantization']; oscale, ozp = out['quantization']
    preds = []
    for s in range(0, len(X), BATCH):
        xb = X[s:s + BATCH]
        if inp['dtype'] == np.int8:
            xb = np.clip(np.round(xb / scale + zp), -128, 127).astype(np.int8)
        it.resize_tensor_input(inp['index'], xb.shape); it.allocate_tensors()
        it.set_tensor(inp['index'], xb); it.invoke()
        preds.append(it.get_tensor(out['index']).argmax(1))      # argmax is unaffected by the output scale
    return np.concatenate(preds)

def latency_ms(path, X):
    it = Interpreter(model_path=path); inp = it.get_input_details()[0]
    it.resize_tensor_input(inp['index'], (1,) + X.shape[1:]); it.allocate_tensors()
    scale, zp = inp['quantization']
    xs = X[:N_WARM + N_LAT]
    if inp['dtype'] == np.int8:
        xs = np.clip(np.round(xs / scale + zp), -128, 127).astype(np.int8)
    for i in range(N_WARM):
        it.set_tensor(inp['index'], xs[i:i + 1]); it.invoke()
    ts = []
    for i in range(N_WARM, N_WARM + N_LAT):
        t0 = time.perf_counter(); it.set_tensor(inp['index'], xs[i:i + 1]); it.invoke(); ts.append(time.perf_counter() - t0)
    ts = 1000 * np.array(ts)
    return float(np.percentile(ts, 90)), float(ts.mean())

log(f'INT8 quantization | TensorFlow {tf.__version__} | Python {platform.python_version()} | CPU: {platform.processor() or platform.machine()} | '
    f'{os.cpu_count()} cores | GPU: {[g.name for g in tf.config.list_physical_devices("GPU")] or "none"}')

In [ ]:
# %% Cell 2 — convert, quantize and evaluate (results are saved per model; re-running skips finished models)
cache = {}
for cfg, kind, path in MODELS:
    name = f'{kind}_{cfg}'; rpath = OUT + f'results/{name}.json'
    if os.path.exists(rpath):
        log(f'{name}: already done, skipping'); continue
    if cache.get('cfg') != cfg:
        cache.clear(); gc.collect()
        Xtr = np.load(FEAT + cfg + '/X_train.npy', mmap_mode='r'); ytr = np.load(FEAT + cfg + '/y_train.npy')
        idx, _ = train_test_split(np.arange(len(ytr)), train_size=N_REP, stratify=ytr, random_state=SEED)
        rep = np.asarray(Xtr[np.sort(idx)], dtype=np.float32)[..., None]
        Xte = np.load(FEAT + cfg + '/X_test.npy').astype(np.float32)[..., None]; yte = np.load(FEAT + cfg + '/y_test.npy').astype(np.int64)
        cache = dict(cfg=cfg, rep=rep, Xte=Xte, yte=yte); del Xtr
        log(f'{cfg}: representative set {len(rep)} training records (stratified) | test {len(yte):,} records')
    c = cache
    model = keras.models.load_model(path, compile=False)
    params = int(sum(np.prod(v.shape) for v in model.trainable_weights))
    log(f'--- {name}: {params:,} parameters | Keras file {os.path.getsize(path) / 1024:.1f} KB ---')
    # Keras float32 reference
    t0 = time.time(); p_keras = model.predict(c['Xte'], batch_size=BATCH, verbose=0).argmax(1)
    m_keras = metrics(c['yte'], p_keras); log(f'{name} | Keras float32: Macro F1 {m_keras["macro_f1"]:.4f} | accuracy {m_keras["accuracy"]:.4f} ({time.time() - t0:.0f}s)')
    # TFLite float32
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    f32 = conv.convert(); p32 = OUT + f'models/{name}_float32.tflite'; open(p32, 'wb').write(f32)
    # TFLite full-integer INT8
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = lambda: ([c['rep'][i:i + 1]] for i in range(len(c['rep'])))
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
    i8 = conv.convert(); p8 = OUT + f'models/{name}_int8.tflite'; open(p8, 'wb').write(i8)
    log(f'{name} | TFLite float32 {len(f32) / 1024:.1f} KB | TFLite INT8 {len(i8) / 1024:.1f} KB ({100 * (1 - len(i8) / len(f32)):.1f}% smaller)')
    t0 = time.time(); m32 = metrics(c['yte'], tflite_predict(p32, c['Xte']))
    log(f'{name} | TFLite float32: Macro F1 {m32["macro_f1"]:.4f} | accuracy {m32["accuracy"]:.4f} ({time.time() - t0:.0f}s)')
    t0 = time.time(); p_i8 = tflite_predict(p8, c['Xte']); m8 = metrics(c['yte'], p_i8)
    agree = float((p_i8 == p_keras).mean())
    log(f'{name} | TFLite INT8:    Macro F1 {m8["macro_f1"]:.4f} | accuracy {m8["accuracy"]:.4f} | same prediction as float32 for {100 * agree:.2f}% of records ({time.time() - t0:.0f}s)')
    log(f'{name} | per-class F1 float32 {m_keras["per_class_f1"]}')
    log(f'{name} | per-class F1 INT8    {m8["per_class_f1"]}')
    (l32, m32l), (l8, m8l) = latency_ms(p32, c['Xte']), latency_ms(p8, c['Xte'])
    log(f'{name} | CPU latency per record (single-stream, {N_LAT} queries): 90th percentile float32 {l32:.4f} ms | INT8 {l8:.4f} ms '
        f'(mean {m32l:.4f} / {m8l:.4f} ms)')
    res = dict(model=name, config=cfg, kind=kind, params=params, keras_kb=round(os.path.getsize(path) / 1024, 1),
               tflite_float32_kb=round(len(f32) / 1024, 1), tflite_int8_kb=round(len(i8) / 1024, 1),
               keras_float32=m_keras, tflite_float32=m32, tflite_int8=m8, int8_agreement_with_float32=agree,
               latency_p90_ms_float32=l32, latency_p90_ms_int8=l8, latency_mean_ms_float32=m32l, latency_mean_ms_int8=m8l, representative_records=N_REP, test_records=int(len(c['yte'])))
    json.dump(res, open(rpath, 'w'), indent=2)
    del model; gc.collect()
log('INT8 quantization: done')

In [ ]:
# %% Cell 3 — summary table
rows = []
for cfg, kind, _ in MODELS:
    p = OUT + f'results/{kind}_{cfg}.json'
    if os.path.exists(p):
        r = json.load(open(p))
        rows.append(dict(configuration=cfg, model=kind, params=r['params'], keras_kb=r['keras_kb'],
                         tflite_f32_kb=r['tflite_float32_kb'], int8_kb=r['tflite_int8_kb'],
                         f1_float32=r['keras_float32']['macro_f1'], f1_int8=r['tflite_int8']['macro_f1'],
                         f1_change=r['tflite_int8']['macro_f1'] - r['keras_float32']['macro_f1'],
                         acc_float32=r['keras_float32']['accuracy'], acc_int8=r['tflite_int8']['accuracy'],
                         agreement=r['int8_agreement_with_float32'], p90_ms_f32=r['latency_p90_ms_float32'], p90_ms_int8=r['latency_p90_ms_int8']))
tab = pd.DataFrame(rows)
pd.set_option('display.width', 250)
print(tab.round(4).to_string(index=False))
json.dump(dict(table=rows), open(OUT + 'summary.json', 'w'), indent=2)